# Day 30 · Your evals on a managed platform

**First-principles question:** Does a vendor's evaluation service agree with your calibrated judge, and what does it add?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import pandas as pd
labels = pd.read_csv(ROOT / "3-evals-that-run-without-you" / "01-judge-calibration" / "labels.csv").fillna("")
labels = labels[labels["you"].isin(["pass", "fail"])]
print(len(labels), "labelled replies from step 1")

## 1 · Estimate before you spend

In [ ]:
CAP_INR = 500
# TODO: from the vendor's pricing page: the cost of grading one reply (judge tokens in and out × price)
COST_PER_REPLY_INR = None
expected_inr = COST_PER_REPLY_INR * len(labels)
print(f"Expected ₹{expected_inr:.0f} (cap ₹{CAP_INR})")

## 2 · The vendor evaluator
Use one built-in evaluator (Azure AI Evaluation SDK, AgentCore Evaluations or Google Agent Evaluations). Keys stay in `.env`.

In [ ]:
def vendor_verdict(row):
    """'pass' or 'fail' from the vendor's evaluator for one reply."""
    # TODO: call the evaluator; turn its score into pass/fail with a threshold you write down
    ...

labels["vendor"] = [vendor_verdict(r) for r in labels.itertuples()]

## 3 · Three-way agreement

In [ ]:
def kappa(a, b):
    a, b = list(a), list(b)
    obs = sum(x == y for x, y in zip(a, b)) / len(a)
    pa, pb = a.count("pass") / len(a), b.count("pass") / len(b)
    chance = pa * pb + (1 - pa) * (1 - pb)
    return (obs - chance) / (1 - chance) if chance < 1 else 1.0

pairs = {"you × judge": kappa(labels["you"], labels["judge"]), "you × vendor": kappa(labels["you"], labels["vendor"]),
         "judge × vendor": kappa(labels["judge"], labels["vendor"])}
pd.Series(pairs).round(2)

**What does the platform give you, and where is your own judge still better?** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert expected_inr <= CAP_INR / 5, "Step 1: estimate under a fifth of your cap before you run"
assert labels["vendor"].isin(["pass", "fail"]).all(), "Step 2: a verdict for every reply"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")